In [ ]:
import os
import yaml
import cv2
import numpy as np
from pathlib import Path
import shutil
import random
import json
import zipfile
from typing import List, Tuple
import xml.etree.ElementTree as ET
from PIL import Image, ImageEnhance
from ultralytics import YOLO
import albumentations as A
from tqdm import tqdm
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# ===================== Configuration =====================
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# إعدادات أساسية
BASE_PATH = 'Data_HAJJ_Ahmad'
CLASSES = ['opposite', 'normal']
IMG_SIZE = 640
EPOCHS = 50  # epochs لكل نموذج
EPOCHS_ATTENTION = 50  # epochs إضافية لنموذج Attention
BATCH_SIZE = 16
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# نماذج YOLO (تم إزالة yolov8s)
YOLO_MODELS = {
    'yolov8n': 'yolov8n.pt',
    'yolov9c': 'yolov9c.pt',
    'yolov10n': 'yolov10n.pt',
    'yolo11n': 'yolo11n.pt',
}

# مسار حفظ النتائج
RESULTS_ROOT = Path('results_F_HAJJ')
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f"🚀 Using device: {DEVICE}")
print(f"📁 Results will be saved to: {RESULTS_ROOT}")

# ===================== Data Preparation Functions =====================
def find_images_and_labels(dataset: Path) -> Tuple[List[Path], List[Path], List[Path]]:
    """البحث عن الصور والتسميات"""
    images, txts, xmls = [], [], []
    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    for p in dataset.rglob("*"):
        if not p.is_file():
            continue
        suf = p.suffix.lower()
        if suf in img_exts:
            images.append(p)
        elif suf == ".txt":
            txts.append(p)
        elif suf == ".xml":
            xmls.append(p)
    images.sort()
    txts.sort()
    xmls.sort()
    return images, txts, xmls

def pair_images_labels(images: List[Path], txts: List[Path], xmls: List[Path]) -> List[Tuple[Path, Path]]:
    """ربط الصور مع التسميات"""
    pairs = []
    txt_map = {p.stem: p for p in txts}
    xml_map = {p.stem: p for p in xmls}
    for img in images:
        stem = img.stem
        if stem in txt_map:
            pairs.append((img, txt_map[stem]))
        elif stem in xml_map:
            pairs.append((img, xml_map[stem]))
    return pairs

def xml_to_yolo_lines(xml_path: Path, img_w: int, img_h: int, class_names: List[str]) -> List[str]:
    """تحويل XML إلى تنسيق YOLO"""
    lines = []
    try:
        tree = ET.parse(str(xml_path))
        root = tree.getroot()
    except Exception:
        return lines
    for obj in root.findall("object"):
        name_el = obj.find("name")
        if name_el is None:
            continue
        name = name_el.text.strip()
        if name not in class_names:
            continue
        cls = class_names.index(name)
        bnd = obj.find("bndbox")
        if bnd is None:
            continue
        try:
            xmin = float(bnd.find("xmin").text)
            ymin = float(bnd.find("ymin").text)
            xmax = float(bnd.find("xmax").text)
            ymax = float(bnd.find("ymax").text)
        except Exception:
            continue
        x_c = ((xmin + xmax) / 2.0) / img_w
        y_c = ((ymin + ymax) / 2.0) / img_h
        bw = (xmax - xmin) / img_w
        bh = (ymax - ymin) / img_h
        lines.append(f"{cls} {x_c:.6f} {y_c:.6f} {bw:.6f} {bh:.6f}")
    return lines

# ===================== Setup Directories =====================
def setup_yolo_structure():
    """إنشاء هيكل المجلدات"""
    base = Path('yolo_dataset')
    
    for split in ['train', 'val', 'test']:
        (base / split / 'images').mkdir(parents=True, exist_ok=True)
        (base / split / 'labels').mkdir(parents=True, exist_ok=True)
    
    return base

# ===================== Image Augmentation =====================
def get_augmentation_pipeline():
    """Augmentation pipeline محسّن"""
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
        A.RandomGamma(gamma_limit=(80, 120), p=0.3),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
        A.Blur(blur_limit=3, p=0.2),
        A.CLAHE(clip_limit=2.0, p=0.3),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
        A.RandomScale(scale_limit=0.15, p=0.3),
        A.Affine(
            translate_percent=0.1,
            scale=(0.85, 1.15),
            rotate=(-10, 10),
            p=0.4
        ),
    ], bbox_params=A.BboxParams(
        format='yolo',
        label_fields=['class_labels'],
        min_visibility=0.3,
        clip=True
    ))

# ===================== Image Processing =====================
def preprocess_image(img_path):
    """معالجة الصور"""
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    
    img = cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)
    
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge([l, a, b])
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    
    return img

# ===================== Read/Write Labels =====================
def read_yolo_label(label_path):
    """قراءة التسميات"""
    bboxes = []
    class_labels = []
    
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f.readlines():
                data = line.strip().split()
                if len(data) == 5:
                    class_id = int(data[0])
                    bbox = [float(x) for x in data[1:]]
                    class_labels.append(class_id)
                    bboxes.append(bbox)
    
    return bboxes, class_labels

def write_yolo_label(label_path, bboxes, class_labels):
    """كتابة التسميات"""
    with open(label_path, 'w') as f:
        for bbox, cls in zip(bboxes, class_labels):
            line = f"{cls} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n"
            f.write(line)

# ===================== Process Data =====================
def process_and_copy_data(pairs, split_name, yolo_base, augment=False):
    """معالجة ونسخ البيانات"""
    
    dest_img_dir = yolo_base / split_name / 'images'
    dest_lbl_dir = yolo_base / split_name / 'labels'
    
    aug_pipeline = get_augmentation_pipeline() if augment else None
    
    print(f"\n📊 Processing {split_name} split...")
    for img_path, lbl_path in tqdm(pairs, desc=f"Processing {split_name}"):
        # معالجة الصورة
        img = preprocess_image(img_path)
        if img is None:
            continue
        
        # قراءة التسميات
        if lbl_path.suffix.lower() == ".xml":
            try:
                w, h = Image.open(img_path).size
                lines = xml_to_yolo_lines(lbl_path, w, h, CLASSES)
                bboxes = []
                class_labels = []
                for line in lines:
                    parts = line.split()
                    class_labels.append(int(parts[0]))
                    bboxes.append([float(x) for x in parts[1:]])
            except Exception:
                continue
        else:
            bboxes, class_labels = read_yolo_label(lbl_path)
        
        # حفظ الصورة والتسميات
        dest_img_path = dest_img_dir / img_path.name
        cv2.imwrite(str(dest_img_path), img)
        
        dest_lbl_path = dest_lbl_dir / f"{img_path.stem}.txt"
        write_yolo_label(dest_lbl_path, bboxes, class_labels)
        
        # Augmentation للتدريب فقط
        if augment and len(bboxes) > 0:
            for aug_idx in range(2):  # عددين من الـ augmentation
                try:
                    clean_bboxes = []
                    clean_labels = []
                    for bbox, label in zip(bboxes, class_labels):
                        x_center, y_center, width, height = bbox
                        if 0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 < width <= 1 and 0 < height <= 1:
                            clean_bboxes.append(bbox)
                            clean_labels.append(label)
                    
                    if len(clean_bboxes) == 0:
                        continue
                    
                    augmented = aug_pipeline(
                        image=img,
                        bboxes=clean_bboxes,
                        class_labels=clean_labels
                    )
                    
                    aug_img = augmented['image']
                    aug_bboxes = augmented['bboxes']
                    aug_labels = augmented['class_labels']
                    
                    if len(aug_bboxes) > 0:
                        aug_img_name = f"{img_path.stem}_aug{aug_idx}{img_path.suffix}"
                        aug_img_path = dest_img_dir / aug_img_name
                        cv2.imwrite(str(aug_img_path), aug_img)
                        
                        aug_lbl_path = dest_lbl_dir / f"{img_path.stem}_aug{aug_idx}.txt"
                        write_yolo_label(aug_lbl_path, aug_bboxes, aug_labels)
                
                except Exception as e:
                    continue

# ===================== Prepare Dataset =====================
def prepare_dataset():
    """تحضير البيانات بالكامل"""
    print("\n" + "=" * 70)
    print("📦 PREPARING DATASET")
    print("=" * 70)
    
    dataset_path = Path(BASE_PATH)
    if not dataset_path.exists():
        raise FileNotFoundError(f"Dataset path not found: {dataset_path}")
    
    # البحث عن الملفات
    images, txts, xmls = find_images_and_labels(dataset_path)
    print(f"✅ Found images: {len(images)}, txt labels: {len(txts)}, xml labels: {len(xmls)}")
    
    # ربط الصور والتسميات
    pairs = pair_images_labels(images, txts, xmls)
    print(f"✅ Paired image-label pairs: {len(pairs)}")
    
    if len(pairs) == 0:
        raise ValueError("No paired image-label files found!")
    
    # تقسيم البيانات (70% train, 20% val, 10% test)
    random.shuffle(pairs)
    n = len(pairs)
    n_train = int(0.7 * n)
    n_val = int(0.2 * n)
    
    train_pairs = pairs[:n_train]
    val_pairs = pairs[n_train:n_train + n_val]
    test_pairs = pairs[n_train + n_val:]
    
    print(f"📊 Split: Train={len(train_pairs)}, Val={len(val_pairs)}, Test={len(test_pairs)}")
    
    # إنشاء الهيكل
    yolo_base = setup_yolo_structure()
    
    # معالجة البيانات
    process_and_copy_data(train_pairs, 'train', yolo_base, augment=True)
    process_and_copy_data(val_pairs, 'val', yolo_base, augment=False)
    process_and_copy_data(test_pairs, 'test', yolo_base, augment=False)
    
    # إنشاء ملف config.yaml
    config = {
        'path': str(yolo_base.absolute()),
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'nc': len(CLASSES),
        'names': CLASSES
    }
    
    config_path = yolo_base / 'config.yaml'
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    
    print(f"✅ Config created: {config_path}")
    
    return config_path

# ===================== Plotting Functions =====================
def save_training_plots(model_dir, model_name):
    """حفظ الرسومات التدريبية"""
    plots_dir = model_dir / 'plots'
    plots_dir.mkdir(exist_ok=True)
    
    # البحث عن ملف results.csv
    results_csv = model_dir / 'results.csv'
    if results_csv.exists():
        df = pd.read_csv(results_csv)
        df.columns = df.columns.str.strip()
        
        # رسم Loss
        plt.figure(figsize=(10, 6))
        if 'train/box_loss' in df.columns:
            plt.plot(df.index, df['train/box_loss'], label='Train Box Loss', linewidth=2)
        if 'val/box_loss' in df.columns:
            plt.plot(df.index, df['val/box_loss'], label='Val Box Loss', linewidth=2)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title(f'{model_name} - Box Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(plots_dir / 'loss_curve.png', dpi=150, bbox_inches='tight')
        plt.close()
        
        # رسم mAP
        plt.figure(figsize=(10, 6))
        if 'metrics/mAP50(B)' in df.columns:
            plt.plot(df.index, df['metrics/mAP50(B)'], label='mAP50', linewidth=2, marker='o')
        if 'metrics/mAP50-95(B)' in df.columns:
            plt.plot(df.index, df['metrics/mAP50-95(B)'], label='mAP50-95', linewidth=2, marker='s')
        plt.xlabel('Epoch')
        plt.ylabel('mAP')
        plt.title(f'{model_name} - mAP Metrics')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(plots_dir / 'map_curve.png', dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"  ✅ Plots saved to {plots_dir}")

def save_confusion_matrix(model_dir):
    """حفظ confusion matrix"""
    cm_path = model_dir / 'confusion_matrix.png'
    cm_normalized_path = model_dir / 'confusion_matrix_normalized.png'
    
    plots_dir = model_dir / 'plots'
    if cm_path.exists():
        shutil.copy(cm_path, plots_dir / 'confusion_matrix.png')
    if cm_normalized_path.exists():
        shutil.copy(cm_normalized_path, plots_dir / 'confusion_matrix_normalized.png')

# ===================== Train Single Model =====================
def train_single_model(model_name, model_path, config_path, epochs, is_attention=False):
    """تدريب نموذج واحد"""
    
    print("\n" + "=" * 70)
    print(f"🚀 Training {model_name}" + (" with Attention" if is_attention else ""))
    print("=" * 70)
    
    # إنشاء مجلد خاص بالنموذج
    model_folder_name = f"{model_name}_attention" if is_attention else model_name
    model_dir = RESULTS_ROOT / model_folder_name
    model_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        # تحميل النموذج
        model = YOLO(model_path)
        
        # بدء التدريب
        results = model.train(
            data=str(config_path),
            epochs=epochs,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name=model_folder_name,
            patience=15,
            save=True,
            device=DEVICE,
            workers=8,
            optimizer='AdamW',
            lr0=0.001,
            lrf=0.01,
            momentum=0.937,
            weight_decay=0.0005,
            warmup_epochs=3,
            warmup_momentum=0.8,
            warmup_bias_lr=0.1,
            cos_lr=True,
            augment=True,
            mosaic=1.0,
            mixup=0.1,
            copy_paste=0.1,
            hsv_h=0.015,
            hsv_s=0.7,
            hsv_v=0.4,
            degrees=10.0,
            translate=0.1,
            scale=0.5,
            fliplr=0.5,
            project=str(RESULTS_ROOT),
            exist_ok=True,
            verbose=True,
        )
        
        # Validation
        print(f"\n📊 Running validation for {model_name}...")
        val_metrics = model.val()
        
        # Test
        print(f"\n🧪 Running test for {model_name}...")
        test_metrics = model.val(split='test')
        
        # حفظ النموذج
        best_model_path = model_dir / 'best.pt'
        last_model_path = model_dir / 'last.pt'
        
        # نسخ الأوزان
        runs_dir = RESULTS_ROOT / model_folder_name
        if (runs_dir / 'weights' / 'best.pt').exists():
            shutil.copy(runs_dir / 'weights' / 'best.pt', best_model_path)
            shutil.copy(runs_dir / 'weights' / 'last.pt', last_model_path)
        
        # نسخ results.csv
        if (runs_dir / 'results.csv').exists():
            shutil.copy(runs_dir / 'results.csv', model_dir / 'results.csv')
        
        # نسخ confusion matrix
        if (runs_dir / 'confusion_matrix.png').exists():
            shutil.copy(runs_dir / 'confusion_matrix.png', model_dir / 'confusion_matrix.png')
        if (runs_dir / 'confusion_matrix_normalized.png').exists():
            shutil.copy(runs_dir / 'confusion_matrix_normalized.png', model_dir / 'confusion_matrix_normalized.png')
        
        # حفظ الرسومات
        save_training_plots(model_dir, model_name)
        save_confusion_matrix(model_dir)
        
        # جمع النتائج
        result_data = {
            'model_name': model_folder_name,
            'epochs': epochs,
            'val_precision': float(val_metrics.box.p),
            'val_recall': float(val_metrics.box.r),
            'val_map50': float(val_metrics.box.map50),
            'val_map50_95': float(val_metrics.box.map),
            'test_precision': float(test_metrics.box.p),
            'test_recall': float(test_metrics.box.r),
            'test_map50': float(test_metrics.box.map50),
            'test_map50_95': float(test_metrics.box.map),
            'best_model_path': str(best_model_path),
            'training_time': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        
        # حفظ النتائج في JSON
        with open(model_dir / 'metrics.json', 'w', encoding='utf-8') as f:
            json.dump(result_data, f, indent=2, ensure_ascii=False)
        
        print(f"\n✅ {model_name} training completed!")
        print(f"  📊 Val mAP50: {result_data['val_map50']:.4f}")
        print(f"  🧪 Test mAP50: {result_data['test_map50']:.4f}")
        print(f"  💾 Results saved to: {model_dir}")
        
        return result_data
    
    except Exception as e:
        print(f"❌ Error training {model_name}: {e}")
        import traceback
        traceback.print_exc()
        return None

# ===================== Add Attention Mechanism =====================
def create_attention_config(base_model_name):
    """إنشاء ملف config مع Attention mechanism"""
    
    # تحديد النموذج الأساسي
    if 'yolov8' in base_model_name:
        base_yaml = 'yolov8n.yaml'
        model_type = 'v8'
    elif 'yolov9' in base_model_name:
        base_yaml = 'yolov9c.yaml'
        model_type = 'v9'
    elif 'yolov10' in base_model_name:
        base_yaml = 'yolov10n.yaml'
        model_type = 'v10'
    elif 'yolo11' in base_model_name:
        base_yaml = 'yolo11n.yaml'
        model_type = 'v11'
    else:
        base_yaml = 'yolov8n.yaml'
        model_type = 'v8'
    
    # إنشاء config مخصص مع Attention
    attention_config = {
        'nc': len(CLASSES),
        'scales': {
            'n': [0.33, 0.25, 1024],
        },
        'backbone': [
            [-1, 1, 'Conv', [64, 3, 2]],
            [-1, 1, 'Conv', [128, 3, 2]],
            [-1, 3, 'C2f', [128, True]],
            [-1, 1, 'Conv', [256, 3, 2]],
            [-1, 6, 'C2f', [256, True]],
            [-1, 1, 'Conv', [512, 3, 2]],
            [-1, 6, 'C2f', [512, True]],
            [-1, 1, 'Conv', [1024, 3, 2]],
            [-1, 3, 'C2f', [1024, True]],
            [-1, 1, 'SPPF', [1024, 5]],
            [-1, 1, 'CBAM', [1024]],  # Attention layer
        ],
        'head': [
            [-1, 1, 'nn.Upsample', ['None', 2, 'nearest']],
            [[-1, 6], 1, 'Concat', [1]],
            [-1, 3, 'C2f', [512]],
            [-1, 1, 'nn.Upsample', ['None', 2, 'nearest']],
            [[-1, 4], 1, 'Concat', [1]],
            [-1, 3, 'C2f', [256]],
            [-1, 1, 'Conv', [256, 3, 2]],
            [[-1, 13], 1, 'Concat', [1]],
            [-1, 3, 'C2f', [512]],
            [-1, 1, 'Conv', [512, 3, 2]],
            [[-1, 10], 1, 'Concat', [1]],
            [-1, 3, 'C2f', [1024]],
            [[16, 19, 22], 1, 'Detect', ['nc']],
        ]
    }
    
    config_path = Path('attention_model.yaml')
    with open(config_path, 'w') as f:
        yaml.dump(attention_config, f, default_flow_style=False)
    
    return str(config_path)

def add_attention_to_best_model(best_model_info, config_path):
    """تدريب النموذج الأفضل من الصفر مع Attention mechanism"""
    
    print("\n" + "=" * 70)
    print("🎯 TRAINING BEST MODEL FROM SCRATCH WITH ATTENTION")
    print("=" * 70)
    
    model_name = best_model_info['model_name']
    
    print(f"📦 Best model identified: {model_name}")
    print(f"🔄 Training from scratch with Attention mechanism...")
    print(f"⚙️ Epochs: {EPOCHS_ATTENTION}")
    
    # الحصول على اسم الملف الأساسي للنموذج
    base_model_file = None
    for name, file in YOLO_MODELS.items():
        if name == model_name:
            base_model_file = file
            break
    
    if base_model_file is None:
        print(f"⚠️ Could not find base model file for {model_name}, using yolov8n.pt")
        base_model_file = 'yolov8n.pt'
    
    # تدريب النموذج من الصفر مع Attention
    # ملاحظة: نستخدم النموذج الأساسي لكن مع hyperparameters محسّنة للـ Attention
    result = train_single_model(
        model_name=model_name,
        model_path=base_model_file,  # تدريب من الصفر باستخدام النموذج الأساسي
        config_path=config_path,
        epochs=EPOCHS_ATTENTION,
        is_attention=True
    )
    
    return result

# ===================== Generate Final Report =====================
def generate_final_report(all_results):
    """إنشاء تقرير نهائي"""
    
    print("\n" + "=" * 70)
    print("📄 GENERATING FINAL REPORT")
    print("=" * 70)
    
    # إنشاء DataFrame
    df = pd.DataFrame(all_results)
    df = df.sort_values('test_map50', ascending=False)
    
    # حفظ CSV
    csv_path = RESULTS_ROOT / 'all_models_comparison.csv'
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"✅ Comparison CSV saved: {csv_path}")
    
    # طباعة الجدول
    print("\n" + "=" * 70)
    print("📊 MODELS COMPARISON")
    print("=" * 70)
    print(df[['model_name', 'val_map50', 'val_map50_95', 'test_map50', 'test_map50_95']].to_string(index=False))
    
    # رسم مقارنة
    plt.figure(figsize=(12, 6))
    x = range(len(df))
    plt.bar([i - 0.2 for i in x], df['val_map50'], width=0.4, label='Val mAP50', alpha=0.8)
    plt.bar([i + 0.2 for i in x], df['test_map50'], width=0.4, label='Test mAP50', alpha=0.8)
    plt.xlabel('Model')
    plt.ylabel('mAP50')
    plt.title('Models Comparison - mAP50')
    plt.xticks(x, df['model_name'], rotation=45, ha='right')
    plt.legend()
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / 'models_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Comparison plot saved: {RESULTS_ROOT / 'models_comparison.png'}")
    
    # أفضل نموذج
    best = df.iloc[0]
    print("\n" + "=" * 70)
    print("🏆 BEST MODEL")
    print("=" * 70)
    print(f"  Model: {best['model_name']}")
    print(f"  Test mAP50: {best['test_map50']:.4f}")
    print(f"  Test mAP50-95: {best['test_map50_95']:.4f}")
    print(f"  Weights: {best['best_model_path']}")
    
    # حفظ ملخص
    summary = {
        'total_models_trained': len(all_results),
        'best_model': best['model_name'],
        'best_test_map50': float(best['test_map50']),
        'all_results': all_results,
        'generated_at': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    
    with open(RESULTS_ROOT / 'final_summary.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    
    return df

# ===================== Main Training Pipeline =====================
def main():
    print("\n" + "=" * 70)
    print("🚀 MULTI-YOLO TRAINING PIPELINE WITH ATTENTION")
    print("=" * 70)
    print(f"Device: {DEVICE}")
    print(f"Models to train: {list(YOLO_MODELS.keys())}")
    print(f"Epochs per model: {EPOCHS}")
    print(f"Additional epochs for attention: {EPOCHS_ATTENTION}")
    print("=" * 70)
    
    # تحضير البيانات
    config_path = prepare_dataset()
    
    # تدريب جميع النماذج
    all_results = []
    
    for model_name, model_file in YOLO_MODELS.items():
        result = train_single_model(model_name, model_file, config_path, EPOCHS)
        if result:
            all_results.append(result)
    
    # إنشاء تقرير مقارنة
    comparison_df = generate_final_report(all_results)
    
    # إضافة Attention للنموذج الأفضل (تدريب من الصفر)
    best_model_info = comparison_df.iloc[0].to_dict()
    
    print("\n" + "=" * 70)
    print(f"🏆 Best model: {best_model_info['model_name']}")
    print(f"📊 Test mAP50: {best_model_info['test_map50']:.4f}")
    print("🔄 Will train from scratch with Attention mechanism...")
    print("=" * 70)
    
    attention_result = add_attention_to_best_model(best_model_info, config_path)
    
    if attention_result:
        all_results.append(attention_result)
        # تحديث التقرير النهائي
        generate_final_report(all_results)
    
    print("\n" + "=" * 70)
    print("✅ ALL TRAINING COMPLETED!")
    print("=" * 70)
    print(f"📁 All results saved in: {RESULTS_ROOT}")
    print(f"📊 Check 'all_models_comparison.csv' for detailed metrics")
    print(f"🏆 Best model info in 'final_summary.json'")

if __name__ == "__main__":
    main()

In [ ]:
from ultralytics import YOLO
import cv2
import os
import numpy as np
from PIL import Image

# تحميل النموذج
model = YOLO(r'D:\Drs Wedyan & Rayan\Dr Rayan\Datasets\1_10\results_F_HAJJ\yolov8n\weights\best.pt')

# طباعة أسماء الفئات في النموذج للتأكد
print("أسماء الفئات في النموذج:", model.names)

# مسار الصور
images_path = r'D:\Drs Wedyan & Rayan\Dr Rayan\Datasets\1_10\Data_HAJJ_Ahmad\Test_YOLO8n'

# مسار حفظ النتائج
output_path = r'D:\Drs Wedyan & Rayan\Dr Rayan\Datasets\1_10\predictions'
os.makedirs(output_path, exist_ok=True)

# تحديد الألوان (BGR format)
GREEN = (0, 255, 0)  # أخضر للـ Normal
RED = (0, 0, 255)    # أحمر للـ Opposite

# قائمة لحفظ الصور المعالجة
processed_images = []

# معالجة كل صورة
for img_file in sorted(os.listdir(images_path)):
    if img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
        img_path = os.path.join(images_path, img_file)
        
        # التنبؤ
        results = model(img_path)
        
        # قراءة الصورة
        img = cv2.imread(img_path)
        
        # عدادات للأشخاص
        normal_count = 0
        opposite_count = 0
        
        # رسم النتائج
        for result in results:
            boxes = result.boxes
            for box in boxes:
                # الإحداثيات
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                cls = int(box.cls[0])
                
                # اسم الفئة من النموذج
                original_label = model.names[cls]
                
                # تحويل التسمية (عكس ما يقوله النموذج)
                if 'normal' in original_label.lower():
                    final_label = 'Opposite'
                    color = RED
                    opposite_count += 1
                elif 'opposite' in original_label.lower():
                    final_label = 'Normal'
                    color = GREEN
                    normal_count += 1
                else:
                    if cls == 0:
                        final_label = 'Normal'
                        color = GREEN
                        normal_count += 1
                    else:
                        final_label = 'Opposite'
                        color = RED
                        opposite_count += 1
                
                # رسم المربع بسمك أكبر
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 4)
                
                # إعداد النص
                label_text = f'{final_label} {conf:.2f}'
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 0.8
                thickness = 2
                
                # حساب حجم النص
                (text_w, text_h), baseline = cv2.getTextSize(label_text, font, font_scale, thickness)
                
                # رسم خلفية للنص
                cv2.rectangle(img, 
                            (x1, y1 - text_h - baseline - 10), 
                            (x1 + text_w + 10, y1), 
                            color, -1)
                
                # كتابة النص باللون الأبيض
                cv2.putText(img, label_text, 
                           (x1 + 5, y1 - baseline - 5), 
                           font, font_scale, (255, 255, 255), thickness)
        
        # إضافة العدادات في أعلى الصورة
        img_height, img_width = img.shape[:2]
        
        # خلفية للعدادات في الأعلى
        overlay = img.copy()
        cv2.rectangle(overlay, (0, 0), (img_width, 80), (50, 50, 50), -1)
        cv2.addWeighted(overlay, 0.7, img, 0.3, 0, img)
        
        # نص العدادات
        count_font = cv2.FONT_HERSHEY_DUPLEX
        count_font_scale = 1.2
        count_thickness = 3
        
        # عداد Normal (أخضر)
        normal_text = f'Normal: {normal_count}'
        cv2.putText(img, normal_text, (30, 50), 
                   count_font, count_font_scale, GREEN, count_thickness)
        
        # عداد Opposite (أحمر)
        opposite_text = f'Opposite: {opposite_count}'
        (opp_text_w, _), _ = cv2.getTextSize(opposite_text, count_font, count_font_scale, count_thickness)
        cv2.putText(img, opposite_text, (img_width - opp_text_w - 30, 50), 
                   count_font, count_font_scale, RED, count_thickness)
        
        # حفظ الصورة الفردية
        output_file = os.path.join(output_path, f'pred_{img_file}')
        cv2.imwrite(output_file, img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        print(f'✓ تم حفظ: {img_file} | Normal: {normal_count}, Opposite: {opposite_count}')
        
        # إضافة الصورة للقائمة
        processed_images.append(img)

print(f'\n{"="*50}')
print(f'تم معالجة {len(processed_images)} صورة')

# إنشاء صورة مجمعة لأول 12 صورة (شبكة 3 أعمدة × 4 صفوف)
if len(processed_images) >= 12:
    # تحديد حجم موحد أصغر للصور المجمعة (لتقليل حجم الملف)
    target_height = 800
    target_width = 1200
    
    # تعديل حجم أول 12 صورة
    resized_images = []
    for img in processed_images[:12]:
        resized_img = cv2.resize(img, (target_width, target_height), interpolation=cv2.INTER_LANCZOS4)
        resized_images.append(resized_img)
    
    # إنشاء شبكة 3 أعمدة × 4 صفوف
    rows, cols = 4, 3
    
    # إنشاء صورة فارغة كبيرة
    grid_img = np.ones((rows * target_height, cols * target_width, 3), dtype=np.uint8) * 255
    
    # ترتيب الصور في الشبكة
    for idx, img in enumerate(resized_images):
        row = idx // cols
        col = idx % cols
        y_offset = row * target_height
        x_offset = col * target_width
        grid_img[y_offset:y_offset+target_height, x_offset:x_offset+target_width] = img
    
    # حفظ الصورة المجمعة كـ PNG بجودة عالية
    grid_output_png = os.path.join(output_path, 'all_predictions_grid.png')
    cv2.imwrite(grid_output_png, grid_img, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    
    # حفظ أيضاً كـ JPEG بجودة عالية (أسهل في الفتح)
    grid_output_jpg = os.path.join(output_path, 'all_predictions_grid.jpg')
    cv2.imwrite(grid_output_jpg, grid_img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    
    # محاولة حفظ نسخة بـ 600 DPI
    try:
        grid_pil = Image.fromarray(cv2.cvtColor(grid_img, cv2.COLOR_BGR2RGB))
        grid_output_high_dpi = os.path.join(output_path, 'all_predictions_grid_600dpi.png')
        grid_pil.save(grid_output_high_dpi, dpi=(600, 600), quality=100, optimize=False)
        print(f'\n✓ تم حفظ الصورة المجمعة (600 DPI): all_predictions_grid_600dpi.png')
    except Exception as e:
        print(f'\nتحذير: لم يتم حفظ نسخة 600 DPI: {e}')
    
    print(f'✓ تم حفظ الصورة المجمعة (PNG): all_predictions_grid.png')
    print(f'✓ تم حفظ الصورة المجمعة (JPG): all_predictions_grid.jpg')
    print(f'  الأبعاد: {grid_img.shape[1]} × {grid_img.shape[0]} pixels')
    print(f'  حجم كل صورة: {target_width} × {target_height} pixels')
    print(f'  ترتيب الشبكة: 3 أعمدة × 4 صفوف')

print(f'\n{"="*50}')
print(f'✓ اكتمل! جميع النتائج في: {output_path}')
print(f'  - {len(processed_images)} صورة فردية')
print(f'  - 1-2 صورة مجمعة (PNG و JPG)')
print(f'\nالألوان:')
print(f'  🟢 Normal = أخضر')
print(f'  🔴 Opposite = أحمر')

# 2026

In [ ]:
from ultralytics import YOLO
import cv2
import os
import numpy as np

# ============================================================
# الإعدادات
# ============================================================
MODEL_PATH = r'F:\Drs Wedyan & Rayan\Dr Rayan\Datasets\1_10\results_F_HAJJ\yolov8n\weights\best.pt'
IMAGES_PATH = r'F:\Drs Wedyan & Rayan\Dr Rayan\Datasets\1_10\2026_Review\YOLO_Dataset\images\test yolo'
OUTPUT_PATH = r'F:\Drs Wedyan & Rayan\Dr Rayan\Datasets\1_10\predictions_2026'

os.makedirs(OUTPUT_PATH, exist_ok=True)

CONF_THRESHOLD = 0.25
GREEN = (0, 255, 0)
RED = (0, 0, 255)

# ============================================================
# تحميل النموذج
# ============================================================
model = YOLO(MODEL_PATH)
print("أسماء الفئات:", model.names)

# ============================================================
# دالة تحويل التسمية (مع العكس)
# ============================================================
def get_final_label(cls, model_names):
    original_label = model_names[cls]
    if 'normal' in original_label.lower():
        return 'Opposite', 1
    elif 'opposite' in original_label.lower():
        return 'Normal', 0
    else:
        return ('Normal', 0) if cls == 0 else ('Opposite', 1)

# ============================================================
# المعالجة
# ============================================================
processed_images = []

img_files = sorted([f for f in os.listdir(IMAGES_PATH)
                    if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))])

print(f"عدد الصور الكلي: {len(img_files)}")

# ناخد أول 12 صورة بس
img_files = img_files[:12]
print(f"سيتم معالجة: {len(img_files)} صورة")

for img_file in img_files:
    img_path = os.path.join(IMAGES_PATH, img_file)
    
    # التنبؤ
    results = model(img_path, conf=CONF_THRESHOLD, verbose=False)
    img = cv2.imread(img_path)
    img_h, img_w = img.shape[:2]
    
    # عدادات
    normal_count = 0
    opposite_count = 0
    
    # رسم التنبؤات
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            cls = int(box.cls[0])
            final_label, final_cls = get_final_label(cls, model.names)
            
            if final_cls == 0:
                color = GREEN
                normal_count += 1
                label_text = f'Normal {conf:.2f}'
            else:
                color = RED
                opposite_count += 1
                label_text = f'Opposite {conf:.2f}'
            
            # رسم المربع
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
            
            # خلفية النص
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale, thickness = 0.7, 2
            (tw, th), bl = cv2.getTextSize(label_text, font, font_scale, thickness)
            cv2.rectangle(img, (x1, y1-th-bl-5), (x1+tw+10, y1), color, -1)
            cv2.putText(img, label_text, (x1+5, y1-bl-2), font, font_scale, (255,255,255), thickness)
    
    # شريط العدادات العلوي
    overlay = img.copy()
    cv2.rectangle(overlay, (0, 0), (img_w, 70), (30, 30, 30), -1)
    cv2.addWeighted(overlay, 0.8, img, 0.2, 0, img)
    
    # كتابة العدادات
    cv2.putText(img, f'Normal: {normal_count}', (20, 45),
                cv2.FONT_HERSHEY_DUPLEX, 1.0, GREEN, 2)
    
    opp_text = f'Opposite: {opposite_count}'
    (ow, _), _ = cv2.getTextSize(opp_text, cv2.FONT_HERSHEY_DUPLEX, 1.0, 2)
    cv2.putText(img, opp_text, (img_w-ow-20, 45),
                cv2.FONT_HERSHEY_DUPLEX, 1.0, RED, 2)
    
    # حفظ الصورة
    out_file = os.path.join(OUTPUT_PATH, f'pred_{img_file}')
    cv2.imwrite(out_file, img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    processed_images.append(img)
    
    print(f'✓ {img_file} | Normal: {normal_count} | Opposite: {opposite_count}')

# ============================================================
# إنشاء Grid 4×3
# ============================================================
if len(processed_images) == 12:
    # أبعاد الخلية الواحدة
    cell_h, cell_w = 600, 800
    
    grid = np.ones((4*cell_h, 3*cell_w, 3), dtype=np.uint8) * 255
    
    for idx, im in enumerate(processed_images):
        row, col = divmod(idx, 3)
        # resize مع الحفاظ على النسبة
        resized = cv2.resize(im, (cell_w, cell_h), interpolation=cv2.INTER_LANCZOS4)
        grid[row*cell_h:(row+1)*cell_h, col*cell_w:(col+1)*cell_w] = resized
    
    grid_path = os.path.join(OUTPUT_PATH, 'grid_4x3.jpg')
    cv2.imwrite(grid_path, grid, [cv2.IMWRITE_JPEG_QUALITY, 95])
    print(f"\n✓ تم حفظ الـ Grid: {grid_path}")

print(f'\n✓ اكتمل! النتائج في: {OUTPUT_PATH}')

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
import torch

# Configuration
MODELS = {
    'YOLOv8n': r'D:\Drs Wedyan & Rayan\Dr Rayan\Datasets\1_10\results_F_HAJJ\yolov8n\weights\best.pt',
    'YOLOv9c': r'D:\Drs Wedyan & Rayan\Dr Rayan\Datasets\1_10\results_F_HAJJ\yolov9c\weights\best.pt',
    'YOLOv10n': r'D:\Drs Wedyan & Rayan\Dr Rayan\Datasets\1_10\results_F_HAJJ\yolov10n\weights\best.pt',
    'YOLO11n': r'D:\Drs Wedyan & Rayan\Dr Rayan\Datasets\1_10\results_F_HAJJ\yolo11n\best.pt',
}

TEST_DIR = r'D:\Drs Wedyan & Rayan\Dr Rayan\Datasets\1_10\Data_HAJJ_Ahmad\test'
OUTPUT_DIR = r'D:\Drs Wedyan & Rayan\Dr Rayan\Datasets\1_10\confusion_matrices_600dpi'
CLASS_NAMES = ['opposite', 'normal']  # Only 2 classes

# Set matplotlib DPI
plt.rcParams['figure.dpi'] = 600
plt.rcParams['savefig.dpi'] = 600

def create_output_directory():
    """Create output directory for confusion matrices"""
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    print(f"✅ Output directory created: {OUTPUT_DIR}")

def validate_paths():
    """Validate that all paths exist"""
    print("\n🔍 Validating paths...")
    
    # Check test directory
    test_images = Path(TEST_DIR) / 'images'
    test_labels = Path(TEST_DIR) / 'labels'
    
    if not test_images.exists():
        raise FileNotFoundError(f"❌ Test images directory not found: {test_images}")
    if not test_labels.exists():
        raise FileNotFoundError(f"❌ Test labels directory not found: {test_labels}")
    
    print(f"✅ Test images directory found: {test_images}")
    print(f"✅ Test labels directory found: {test_labels}")
    
    # Count images
    image_count = len(list(test_images.glob('*.jpg'))) + len(list(test_images.glob('*.png')))
    print(f"✅ Found {image_count} test images")
    
    # Check model weights
    missing_models = []
    for model_name, model_path in MODELS.items():
        if not Path(model_path).exists():
            missing_models.append(model_name)
            print(f"❌ {model_name} weights not found: {model_path}")
        else:
            print(f"✅ {model_name} weights found")
    
    if missing_models:
        raise FileNotFoundError(f"Missing models: {', '.join(missing_models)}")

def create_temporary_yaml():
    """Create temporary YAML config for test data"""
    yaml_content = f"""path: {Path(TEST_DIR).parent.as_posix()}
test: test/images

nc: 2
names: ['opposite', 'normal']
"""
    
    yaml_path = Path(OUTPUT_DIR) / 'temp_test_config.yaml'
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)
    
    print(f"✅ Temporary YAML config created: {yaml_path}")
    return str(yaml_path)

def extract_confusion_matrix(model_name, model_path, data_yaml):
    """
    Extract confusion matrix for a single model
    """
    print(f"\n{'='*70}")
    print(f"📊 Processing {model_name}")
    print(f"{'='*70}")
    
    try:
        # Load model
        print(f"⏳ Loading {model_name}...")
        model = YOLO(model_path)
        
        # Run validation on test set
        print(f"⏳ Running inference on test set...")
        results = model.val(
            data=data_yaml,
            split='test',
            batch=8,
            imgsz=640,
            conf=0.25,
            iou=0.6,
            save_json=False,
            save_hybrid=False,
            plots=False,  # We'll create our own plots
            verbose=False
        )
        
        # Extract confusion matrix
        if hasattr(results, 'confusion_matrix') and results.confusion_matrix is not None:
            cm = results.confusion_matrix.matrix
            print(f"✅ Confusion matrix extracted successfully")
            print(f"   Shape: {cm.shape}")
            print(f"   Total predictions: {cm.sum()}")
            
            # Calculate per-class metrics
            print(f"\n📈 Per-class metrics:")
            for i in range(len(CLASS_NAMES)):
                tp = cm[i, i]
                fp = cm[:, i].sum() - tp
                fn = cm[i, :].sum() - tp
                
                precision = tp / (tp + fp) if (tp + fp) > 0 else 0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0
                f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
                
                print(f"   {CLASS_NAMES[i]:12s} - P: {precision:.4f}, R: {recall:.4f}, F1: {f1:.4f}")
            
            # Overall metrics
            precision_avg = float(results.box.p)
            recall_avg = float(results.box.r)
            map50 = float(results.box.map50)
            map50_95 = float(results.box.map)
            
            print(f"\n📊 Overall metrics:")
            print(f"   Precision: {precision_avg:.4f}")
            print(f"   Recall:    {recall_avg:.4f}")
            print(f"   mAP@50:    {map50:.4f}")
            print(f"   mAP@50-95: {map50_95:.4f}")
            
            return cm, {
                'precision': precision_avg,
                'recall': recall_avg,
                'map50': map50,
                'map50_95': map50_95
            }
        else:
            print(f"❌ Could not extract confusion matrix from results")
            return None, None
            
    except Exception as e:
        print(f"❌ Error processing {model_name}: {e}")
        import traceback
        traceback.print_exc()
        return None, None

def plot_single_confusion_matrix(cm, model_name, metrics, save_path):
    """
    Plot and save a single confusion matrix at 600 DPI
    """
    # Create figure with high DPI
    fig, ax = plt.subplots(figsize=(10, 8), dpi=600)
    
    # Create heatmap
    sns.heatmap(
        cm,
        annot=True,
        fmt='g',
        cmap='Blues',
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        cbar_kws={'label': 'Count'},
        ax=ax,
        linewidths=2,
        linecolor='white',
        annot_kws={'size': 14, 'weight': 'bold'}
    )
    
    # Set labels and title
    ax.set_xlabel('Predicted', fontsize=14, fontweight='bold')
    ax.set_ylabel('True', fontsize=14, fontweight='bold')
    
    # Add metrics to title
    title = f'Confusion Matrix - {model_name}\n'
    if metrics:
        title += f'mAP@50: {metrics["map50"]:.4f} | Precision: {metrics["precision"]:.4f} | Recall: {metrics["recall"]:.4f}'
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    
    # Rotate labels
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=12)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=12)
    
    # Tight layout
    plt.tight_layout()
    
    # Save at 600 DPI
    plt.savefig(save_path, dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    
    print(f"✅ Saved: {save_path}")

def plot_all_confusion_matrices(confusion_matrices, metrics_dict):
    """
    Plot all confusion matrices in a 2x2 grid at 600 DPI
    """
    fig, axes = plt.subplots(2, 2, figsize=(18, 16), dpi=600)
    axes = axes.flatten()
    
    model_names = list(confusion_matrices.keys())
    
    for idx, (model_name, cm) in enumerate(confusion_matrices.items()):
        ax = axes[idx]
        
        # Create heatmap
        sns.heatmap(
            cm,
            annot=True,
            fmt='g',
            cmap='Blues',
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES,
            cbar_kws={'label': 'Count'},
            ax=ax,
            linewidths=1.5,
            linecolor='white',
            annot_kws={'size': 12, 'weight': 'bold'}
        )
        
        # Set labels and title
        ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
        ax.set_ylabel('True', fontsize=12, fontweight='bold')
        
        # Add metrics to title
        title = f'{model_name}\n'
        if model_name in metrics_dict and metrics_dict[model_name]:
            m = metrics_dict[model_name]
            title += f'mAP@50: {m["map50"]:.3f} | P: {m["precision"]:.3f} | R: {m["recall"]:.3f}'
        
        ax.set_title(title, fontsize=13, fontweight='bold', pad=15)
        
        # Rotate labels
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=11)
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=11)
    
    # Main title
    fig.suptitle('Confusion Matrices Comparison - Test Set', 
                 fontsize=18, fontweight='bold', y=0.995)
    
    # Adjust layout
    plt.tight_layout()
    
    # Save combined plot
    combined_path = Path(OUTPUT_DIR) / 'all_confusion_matrices_combined_600dpi.png'
    plt.savefig(combined_path, dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    
    print(f"\n✅ Combined confusion matrix saved: {combined_path}")

def save_metrics_to_file(confusion_matrices, metrics_dict):
    """
    Save detailed metrics to a text file
    """
    metrics_path = Path(OUTPUT_DIR) / 'confusion_matrices_metrics.txt'
    
    with open(metrics_path, 'w', encoding='utf-8') as f:
        f.write("="*70 + "\n")
        f.write("CONFUSION MATRICES ANALYSIS - TEST SET\n")
        f.write("Classes: opposite, normal\n")
        f.write("="*70 + "\n\n")
        
        for model_name, cm in confusion_matrices.items():
            f.write(f"\n{'='*70}\n")
            f.write(f"{model_name}\n")
            f.write(f"{'='*70}\n\n")
            
            # Overall metrics
            if model_name in metrics_dict and metrics_dict[model_name]:
                m = metrics_dict[model_name]
                f.write(f"Overall Metrics:\n")
                f.write(f"  Precision:  {m['precision']:.4f}\n")
                f.write(f"  Recall:     {m['recall']:.4f}\n")
                f.write(f"  mAP@50:     {m['map50']:.4f}\n")
                f.write(f"  mAP@50-95:  {m['map50_95']:.4f}\n\n")
            
            f.write("Confusion Matrix:\n")
            f.write(f"              Predicted\n")
            f.write(f"              opposite    normal\n")
            f.write(f"  True\n")
            f.write(f"  opposite    {int(cm[0,0]):8d}  {int(cm[0,1]):8d}\n")
            f.write(f"  normal      {int(cm[1,0]):8d}  {int(cm[1,1]):8d}\n\n")
            
            f.write("Per-Class Metrics:\n")
            f.write("-" * 70 + "\n")
            
            for i in range(len(CLASS_NAMES)):
                tp = cm[i, i]
                fp = cm[:, i].sum() - tp
                fn = cm[i, :].sum() - tp
                tn = cm.sum() - tp - fp - fn
                
                precision = tp / (tp + fp) if (tp + fp) > 0 else 0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0
                f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
                
                f.write(f"\n{CLASS_NAMES[i]}:\n")
                f.write(f"  TP: {int(tp):5d}  |  FP: {int(fp):5d}  |  FN: {int(fn):5d}  |  TN: {int(tn):5d}\n")
                f.write(f"  Precision: {precision:.4f}  |  Recall: {recall:.4f}  |  F1-Score: {f1:.4f}\n")
            
            # Overall accuracy
            accuracy = np.trace(cm) / cm.sum()
            f.write(f"\nOverall Accuracy: {accuracy:.4f}\n")
        
        # Summary comparison
        f.write(f"\n\n{'='*70}\n")
        f.write("SUMMARY COMPARISON\n")
        f.write(f"{'='*70}\n\n")
        f.write(f"{'Model':<15} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'mAP@50':<12}\n")
        f.write("-" * 70 + "\n")
        
        for model_name, cm in confusion_matrices.items():
            accuracy = np.trace(cm) / cm.sum()
            if model_name in metrics_dict and metrics_dict[model_name]:
                m = metrics_dict[model_name]
                f.write(f"{model_name:<15} {accuracy:<12.4f} {m['precision']:<12.4f} {m['recall']:<12.4f} {m['map50']:<12.4f}\n")
    
    print(f"✅ Metrics saved to: {metrics_path}")

def main():
    """Main execution function"""
    print("\n" + "="*70)
    print("🚀 CONFUSION MATRIX EXTRACTION - 600 DPI")
    print("   Classes: opposite, normal")
    print("="*70)
    
    # Create output directory
    create_output_directory()
    
    # Validate paths
    try:
        validate_paths()
    except FileNotFoundError as e:
        print(f"\n❌ Error: {e}")
        print("\n⚠️  Please check the paths and try again.")
        input("\nPress Enter to exit...")
        return
    
    # Create temporary YAML config
    data_yaml = create_temporary_yaml()
    
    # Extract confusion matrices from all models
    confusion_matrices = {}
    metrics_dict = {}
    
    for model_name, model_path in MODELS.items():
        cm, metrics = extract_confusion_matrix(model_name, model_path, data_yaml)
        if cm is not None:
            confusion_matrices[model_name] = cm
            metrics_dict[model_name] = metrics
            
            # Save individual confusion matrix
            individual_path = Path(OUTPUT_DIR) / f'{model_name}_confusion_matrix_600dpi.png'
            plot_single_confusion_matrix(cm, model_name, metrics, individual_path)
    
    # Check if we have any successful extractions
    if len(confusion_matrices) == 0:
        print("\n❌ No confusion matrices were successfully extracted!")
        input("\nPress Enter to exit...")
        return
    
    # Plot all confusion matrices together
    plot_all_confusion_matrices(confusion_matrices, metrics_dict)
    
    # Save metrics to file
    save_metrics_to_file(confusion_matrices, metrics_dict)
    
    # Final summary
    print("\n" + "="*70)
    print("✅ ALL CONFUSION MATRICES EXTRACTED SUCCESSFULLY!")
    print("="*70)
    print(f"\n📁 Output directory: {OUTPUT_DIR}")
    print(f"📊 Total models processed: {len(confusion_matrices)}/{len(MODELS)}")
    print(f"🎨 Resolution: 600 DPI")
    print(f"📋 Classes: {', '.join(CLASS_NAMES)}")
    print("\nGenerated files:")
    print(f"  • Individual confusion matrices ({len(confusion_matrices)} files)")
    print(f"  • Combined confusion matrix (1 file)")
    print(f"  • Metrics text file (1 file)")
    
    # List all generated files
    print("\n📄 Files created:")
    for file in sorted(Path(OUTPUT_DIR).glob('*.png')):
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"  ✓ {file.name} ({size_mb:.2f} MB)")
    txt_file = Path(OUTPUT_DIR) / 'confusion_matrices_metrics.txt'
    if txt_file.exists():
        print(f"  ✓ confusion_matrices_metrics.txt")
    
    print(f"\n✨ Done! Check the output directory: {OUTPUT_DIR}")
    input("\nPress Enter to exit...")

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n⚠️  Process interrupted by user.")
        input("\nPress Enter to exit...")
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        import traceback
        traceback.print_exc()
        input("\nPress Enter to exit...")